In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('../data/processed/df_cleaned.csv')

with open('../src/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../src/features.pkl', 'rb') as f:
    FEATURES = pickle.load(f)

X = df[FEATURES]
y = df['risk_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

CLASS_NAMES = ['Low', 'Medium', 'High']
PALETTE     = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
ORDER       = ['Low', 'Medium', 'High']

print(f"X_train : {X_train_scaled.shape}")
print(f"X_test  : {X_test_scaled.shape}")
print(f"Features: {len(FEATURES)}")

X_train : (36000, 19)
X_test  : (9000, 19)
Features: 19


In [3]:
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )
}

print("Models defined:")
for name in models:
    print(f"- {name}")

Models defined:
- Random Forest
- XGBoost
- Logistic Regression


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_validate(
        model, X_train_scaled, y_train,
        cv=cv,
        scoring=['accuracy', 'f1_macro', 'roc_auc_ovr_weighted'],
        return_train_score=True
    )

    print(f"\n{name}")
    print(f"{'-'*45}")
    print(f"CV Accuracy: {scores['test_accuracy'].mean():.4f} +- {scores['test_accuracy'].std():.4f}")
    print(f"CV F1 (macro): {scores['test_f1_macro'].mean():.4f} +- {scores['test_f1_macro'].std():.4f}")
    print(f"CV ROC-AUC: {scores['test_roc_auc_ovr_weighted'].mean():.4f} +- {scores['test_roc_auc_ovr_weighted'].std():.4f}")


Random Forest
---------------------------------------------
CV Accuracy: 0.9997 +- 0.0001
CV F1 (macro): 0.9997 +- 0.0001
CV ROC-AUC: 1.0000 +- 0.0000

XGBoost
---------------------------------------------
CV Accuracy: 0.9974 +- 0.0007
CV F1 (macro): 0.9974 +- 0.0007
CV ROC-AUC: 1.0000 +- 0.0000

Logistic Regression
---------------------------------------------
CV Accuracy: 0.9889 +- 0.0008
CV F1 (macro): 0.9889 +- 0.0008
CV ROC-AUC: 0.9999 +- 0.0000
